<a href="https://colab.research.google.com/github/Likith-Reddy25/Summer-Intern/blob/main/codes/Ad_hoc_COV_ZZ_Feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ad-hoc-COV ZZ Feature QKE

In [ ]:

!pip install -q qiskit==1.1.0 qiskit-machine-learning==0.7.2 qiskit-algorithms==0.3.0 --no-deps
!pip install -q qiskit-aer==0.14.2

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityStatevectorKernel

# ── 1. Load Ad-hoc-COV (LCE, 7-qubit instance) ──────────────────
!wget -q https://raw.githubusercontent.com/qiskit-community/prototype-quantum-kernel-training/main/data/dataset_graph7.csv -O dataset_graph7.csv

df = pd.read_csv("dataset_graph7.csv", header=None)
X_all = df.values[:, :-1]            # 128 x 14
y_all = df.values[:, -1].astype(int)  # +1 / -1, 64 per class

N_FEATURES = X_all.shape[1]   # 14
N_QUBITS   = N_FEATURES       # ZZFeatureMap: 1 feature per qubit

# ── 2. Hyperparameter grid (per paper Sec. III-C) ───────────────
C_GRID      = [0.01, 0.1, 1, 10, 100]
LAMBDA_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]   # quantum kernel bandwidth
N_SPLITS_CV = 5
N_REPS      = 30   # set to 2-3 first to sanity-check runtime, then bump to 30

def scale_to_2pi(X_train, X_test):
    """Min-max scale train (fit) and test (transform) into [0, 2π]."""
    mn, mx = X_train.min(axis=0), X_train.max(axis=0)
    rng = np.where(mx - mn == 0, 1.0, mx - mn)  # avoid /0
    X_train_s = (X_train - mn) / rng * (2 * np.pi)
    X_test_s  = (X_test  - mn) / rng * (2 * np.pi)
    return X_train_s, X_test_s

# ── 3. Build ZZFeatureMap + statevector fidelity kernel ─────────
zz_fm = ZZFeatureMap(feature_dimension=N_QUBITS, reps=2, entanglement="linear")
kernel = FidelityStatevectorKernel(feature_map=zz_fm)

def kernel_matrix(kernel, X1, X2, lam):
    """Evaluate quantum kernel with bandwidth λ applied to inputs."""
    return kernel.evaluate(x_vec=lam * X1, y_vec=lam * X2)

# ── 4. Repetition loop ───────────────────────────────────────────
train_acc, train_kap, train_f1 = [], [], []
test_acc,  test_kap,  test_f1  = [], [], []

for rep in range(N_REPS):
    # independent stratified resampling: 32 train / 32 test per class
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.5, stratify=y_all, random_state=rep
    )
    X_train, X_test = scale_to_2pi(X_train_raw, X_test_raw)

    # ---- 5-fold CV grid search over (C, λ) on the TRAIN set only ----
    best_score, best_C, best_lam = -1, None, None
    skf = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=rep)

    for lam in LAMBDA_GRID:
        K_full_train = kernel_matrix(kernel, X_train, X_train, lam)  # 64x64, computed once per λ

        for C in C_GRID:
            fold_scores = []
            for tr_idx, val_idx in skf.split(X_train, y_train):
                K_tr  = K_full_train[np.ix_(tr_idx, tr_idx)]
                K_val = K_full_train[np.ix_(val_idx, tr_idx)]
                svc = SVC(C=C, kernel="precomputed")
                svc.fit(K_tr, y_train[tr_idx])
                fold_scores.append(accuracy_score(y_train[val_idx], svc.predict(K_val)))
            mean_score = np.mean(fold_scores)
            if mean_score > best_score:
                best_score, best_C, best_lam = mean_score, C, lam

    # ---- Refit on full train with best (C, λ), evaluate on test ----
    K_train = kernel_matrix(kernel, X_train, X_train, best_lam)
    K_test  = kernel_matrix(kernel, X_test,  X_train, best_lam)

    final_svc = SVC(C=best_C, kernel="precomputed")
    final_svc.fit(K_train, y_train)

    pred_train = final_svc.predict(K_train)
    pred_test  = final_svc.predict(K_test)

    train_acc.append(accuracy_score(y_train, pred_train))
    train_kap.append(cohen_kappa_score(y_train, pred_train))
    train_f1.append(f1_score(y_train, pred_train, average="macro"))

    test_acc.append(accuracy_score(y_test, pred_test))
    test_kap.append(cohen_kappa_score(y_test, pred_test))
    test_f1.append(f1_score(y_test, pred_test, average="macro"))

    print(f"rep {rep+1:2d}/{N_REPS}  best C={best_C}, λ={best_lam}  "
          f"train_acc={train_acc[-1]:.3f}  test_acc={test_acc[-1]:.3f}")

# ── 5. Report in Table 2 format ─────────────────────────────────
def fmt(vals):
    return f"{np.mean(vals):.3f} ({np.std(vals):.3f})"

results = pd.DataFrame([{
    "Dataset": "Ad-hoc-COV",
    "Feature Map": "ZZFeatureMap",
    "QKT Parameterization": "-",  # plain QKE, no training
    "Train Accuracy": fmt(train_acc),
    "Train Kappa": fmt(train_kap),
    "Train F1 (macro)": fmt(train_f1),
    "Test Accuracy": fmt(test_acc),
    "Test Kappa": fmt(test_kap),
    "Test F1 (macro)": fmt(test_f1),
}])

results

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.6/308.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 12.9 MB/s eta 0:00:00
rep  1/30  best C=100, λ=0.01  train_acc=1.000  test_acc=0.859
rep  2/30  best C=100, λ=0.01  train_acc=1.000  test_acc=0.719
rep  3/30  best C=100, λ=0.001  train_acc=0.781  test_acc=0.781
rep  4/30  best C=100, λ=0.01  train_acc=1.000  test_acc=0.828
rep  5/30  best C=100, λ=0.01  train_acc=1.000  test_acc=0.750
rep  6/30  best C=100, λ=0.01  train_acc=1.000  test_acc=0.859
rep  7/30  best C=100, λ=0.001  train_acc=0.859  test_acc=0.766
rep  8/30

,Dataset,Feature Map,QKT Parameterization,Train Accuracy,Train Kappa,Train F1 (macro),Test Accuracy,Test Kappa,Test F1 (macro)
0,Ad-hoc-COV,ZZFeatureMap,-,0.932 (0.084),0.864 (0.168),0.930 (0.087),0.784 (0.060),0.569 (0.119),0.779 (0.066)


QKT shared 3

In [ ]:
# ── Setup (Colab) ────────────────────────────────────────────────
!pip install -q qiskit==1.1.0 qiskit-machine-learning==0.7.2 qiskit-algorithms==0.3.0 --no-deps
!pip install -q qiskit-aer==0.14.2 joblib

import os
import pickle
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.circuit.library import ZZFeatureMap
from qiskit_algorithms.optimizers import SPSA
from qiskit_machine_learning.kernels import TrainableFidelityStatevectorKernel
from qiskit_machine_learning.kernels.algorithms import QuantumKernelTrainer

# ── 1. Load Ad-hoc-COV (LCE, 7-qubit instance) ──────────────────
!wget -q https://raw.githubusercontent.com/qiskit-community/prototype-quantum-kernel-training/main/data/dataset_graph7.csv -O dataset_graph7.csv

df = pd.read_csv("dataset_graph7.csv", header=None)
X_all = df.values[:, :-1]              # 128 x 14
y_all = df.values[:, -1].astype(int)   # +1 / -1, 64 per class

N_FEATURES = X_all.shape[1]  # 14
N_QUBITS   = N_FEATURES      # ZZFeatureMap: 1 feature per qubit

# ══════════════════════════════════════════════════════════════
# RUN MODE — set PILOT=True first, confirm timing, then switch off
# ══════════════════════════════════════════════════════════════
PILOT = True   # <-- flip to False for the full 30-rep run

if PILOT:
    N_REPS      = 3
    MAXITER     = 30
    LAMBDA_GRID = [0.01, 0.1, 1.0]     # narrow grid for the pilot
else:
    N_REPS      = 30
    MAXITER     = 50                   # reduced from paper's 400 — see notes below
    LAMBDA_GRID = [0.01, 0.1, 1.0]     # replace with values centered on your QKE best_lam

C_GRID      = [0.01, 0.1, 1, 10, 100]
N_SPLITS_CV = 5
N_JOBS      = -1   # parallelize across λ values; set to 1 to disable

CKPT_PATH = "qkt_zz_shared_adhoccov_ckpt.pkl"

def scale_to_2pi(X_train, X_test):
    mn, mx = X_train.min(axis=0), X_train.max(axis=0)
    rng = np.where(mx - mn == 0, 1.0, mx - mn)
    return (X_train - mn) / rng * (2 * np.pi), (X_test - mn) / rng * (2 * np.pi)

# ── 2. Trainable feature map: Vshared(θ) -> ZZFeatureMap(x) ─────
def build_trainable_zz_shared(n_qubits, reps=2):
    """RXYZ(θ1,θ2,θ3) applied identically to every qubit (3 params total),
    followed by ZZFeatureMap. RXYZ(θ1,θ2,θ3) == Qiskit U(θ1, θ3, θ2)."""
    theta = ParameterVector("θ", 3)
    qc = QuantumCircuit(n_qubits)
    for q in range(n_qubits):
        qc.u(theta[0], theta[2], theta[1], q)
    zz = ZZFeatureMap(feature_dimension=n_qubits, reps=reps, entanglement="linear")
    qc.compose(zz, inplace=True)
    return qc, list(theta)

# ── 3. Per-λ training worker (runs in parallel across λ) ────────
def train_for_lambda(lam, X_train, y_train, maxiter):
    fm, theta_params = build_trainable_zz_shared(N_QUBITS, reps=2)
    trainable_kernel = TrainableFidelityStatevectorKernel(
        feature_map=fm, training_parameters=theta_params
    )
    spsa_opt = SPSA(
        maxiter=maxiter,
        second_order=True,
        hessian_delay=25,
        learning_rate=None,
        perturbation=None,
    )
    qkt = QuantumKernelTrainer(
        quantum_kernel=trainable_kernel,
        loss="svc_loss",
        optimizer=spsa_opt,
        initial_point=[0.0, 0.0, 0.0],
    )
    qkt_result = qkt.fit(lam * X_train, y_train)
    opt_kernel = qkt_result.quantum_kernel
    K_full_train = opt_kernel.evaluate(lam * X_train)
    return lam, opt_kernel, K_full_train

# ── 4. Load checkpoint if resuming ───────────────────────────────
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH, "rb") as f:
        ckpt = pickle.load(f)
    train_acc, train_kap, train_f1 = ckpt["train_acc"], ckpt["train_kap"], ckpt["train_f1"]
    test_acc,  test_kap,  test_f1  = ckpt["test_acc"],  ckpt["test_kap"],  ckpt["test_f1"]
    start_rep = len(train_acc)
    print(f"Resuming from checkpoint: {start_rep} reps already done.")
else:
    train_acc, train_kap, train_f1 = [], [], []
    test_acc,  test_kap,  test_f1  = [], [], []
    start_rep = 0

n_params_used = 3  # shared strategy: always 3, regardless of qubit count

# ── 5. Repetition loop ───────────────────────────────────────────
for rep in range(start_rep, N_REPS):
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.5, stratify=y_all, random_state=rep
    )
    X_train, X_test = scale_to_2pi(X_train_raw, X_test_raw)

    # Train θ for each λ in parallel
    lambda_results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(train_for_lambda)(lam, X_train, y_train, MAXITER) for lam in LAMBDA_GRID
)

    best_score, best_C, best_lam, best_kernel = -1, None, None, None
    skf = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=rep)

    for lam, opt_kernel, K_full_train in lambda_results:
        for C in C_GRID:
            fold_scores = []
            for tr_idx, val_idx in skf.split(X_train, y_train):
                K_tr  = K_full_train[np.ix_(tr_idx, tr_idx)]
                K_val = K_full_train[np.ix_(val_idx, tr_idx)]
                svc = SVC(C=C, kernel="precomputed")
                svc.fit(K_tr, y_train[tr_idx])
                fold_scores.append(accuracy_score(y_train[val_idx], svc.predict(K_val)))
            mean_score = np.mean(fold_scores)
            if mean_score > best_score:
                best_score  = mean_score
                best_C      = C
                best_lam    = lam
                best_kernel = opt_kernel

    # ---- Refit on full train with best (C, λ, θ_opt), evaluate on test ----
    K_train = best_kernel.evaluate(best_lam * X_train)
    K_test  = best_kernel.evaluate(best_lam * X_test, best_lam * X_train)

    final_svc = SVC(C=best_C, kernel="precomputed")
    final_svc.fit(K_train, y_train)

    pred_train = final_svc.predict(K_train)
    pred_test  = final_svc.predict(K_test)

    train_acc.append(accuracy_score(y_train, pred_train))
    train_kap.append(cohen_kappa_score(y_train, pred_train))
    train_f1.append(f1_score(y_train, pred_train, average="macro"))

    test_acc.append(accuracy_score(y_test, pred_test))
    test_kap.append(cohen_kappa_score(y_test, pred_test))
    test_f1.append(f1_score(y_test, pred_test, average="macro"))

    print(f"rep {rep+1:2d}/{N_REPS}  best C={best_C}, λ={best_lam}  "
          f"train_acc={train_acc[-1]:.3f}  test_acc={test_acc[-1]:.3f}")

    # checkpoint after every rep
    with open(CKPT_PATH, "wb") as f:
        pickle.dump({
            "train_acc": train_acc, "train_kap": train_kap, "train_f1": train_f1,
            "test_acc": test_acc,   "test_kap": test_kap,   "test_f1": test_f1,
        }, f)

# ── 6. Report in Table 2 format ─────────────────────────────────
def fmt(vals):
    return f"{np.mean(vals):.3f} ({np.std(vals):.3f})"

results = pd.DataFrame([{
    "Dataset": "Ad-hoc-COV",
    "Feature Map": "ZZFeatureMap",
    "QKT Parameterization": f"shared ({n_params_used})",
    "Train Accuracy": fmt(train_acc),
    "Train Kappa": fmt(train_kap),
    "Train F1 (macro)": fmt(train_f1),
    "Test Accuracy": fmt(test_acc),
    "Test Kappa": fmt(test_kap),
    "Test F1 (macro)": fmt(test_f1),
}])

results

rep  1/3  best C=100, λ=0.01  train_acc=1.000  test_acc=0.875
rep  2/3  best C=100, λ=0.01  train_acc=1.000  test_acc=0.719
rep  3/3  best C=1, λ=0.01  train_acc=0.797  test_acc=0.766


,Dataset,Feature Map,QKT Parameterization,Train Accuracy,Train Kappa,Train F1 (macro),Test Accuracy,Test Kappa,Test F1 (macro)
0,Ad-hoc-COV,ZZFeatureMap,shared (3),0.932 (0.096),0.865 (0.192),0.929 (0.100),0.786 (0.065),0.573 (0.131),0.782 (0.067)
